In [ ]:
import os
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.cluster import KMeans
from sklearn.metrics import (
    normalized_mutual_info_score,
    adjusted_rand_score,
    silhouette_score,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

# Robust path detection: search upward from current working directory for a folder
def find_project_root_with_embeddings(start=Path.cwd()):
    for p in [start] + list(start.parents):
        emb = p / 'Embeddings'
        if emb.exists() and emb.is_dir():
            fe = emb / 'filtered_embeddings'
            pe = emb / 'poem_embeddings'
            if (fe.exists() and any(fe.glob('*.npy'))) or (pe.exists() and any(pe.glob('*.npy'))):
                return p
    return None

project_root = find_project_root_with_embeddings()
if project_root is None:
    # fallback to repository-like default if present on disk
    candidate = Path('/Users/lorene/Desktop/SNLP-Project-main')
    if candidate.exists():
        project_root = candidate
    else:
        project_root = Path.cwd()

BASE_DIR = project_root / 'Embeddings'
FILTERED_CSV = BASE_DIR / 'filtered_poems.csv'
FILTERED_EMBED_DIR = BASE_DIR / 'filtered_embeddings'
POEM_EMBED_DIR = BASE_DIR / 'poem_embeddings'
RESULTS_DIR = BASE_DIR / 'results' / 'cluster_analysis'
FIG_DIR = RESULTS_DIR / 'figures'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# Parameters
K_MIN = 2
K_MAX = 30
SILHOUETTE_SAMPLE = 1500     # maximum sample size for silhouette computation
PAIRWISE_SAMPLE = 5000       # sample size for pairwise similarity analysis
RANDOM_STATE = 42

# Show resolved locations for debugging
print('Resolved project_root:', project_root)
print('Using BASE_DIR:', BASE_DIR)
print('Filtered csv exists:', FILTERED_CSV.exists())
print('Filtered embed dir exists:', FILTERED_EMBED_DIR.exists())
print('Poem embed dir exists:', POEM_EMBED_DIR.exists())


Resolved project_root: /Users/lorene/Desktop/SNLP-Project-main
Using BASE_DIR: /Users/lorene/Desktop/SNLP-Project-main/Embeddings
Filtered csv exists: True
Filtered embed dir exists: True
Poem embed dir exists: True


In [7]:
# Utility functions: load embeddings, normalize, silhouette-safe, k-search, plotting
from sklearn.utils import check_random_state

# Choose embedding directory if not explicitly provided. Prefer filtered embeddings
def choose_embed_dir(filtered=FILTERED_EMBED_DIR, poem=POEM_EMBED_DIR):
    if filtered.exists() and any(filtered.glob('*.npy')):
        return filtered
    if poem.exists() and any(poem.glob('*.npy')):
        return poem
    return None

def load_all_embeddings(emb_dir=None):
    """Load all .npy files from emb_dir (or auto-detect) and return a dict mapping filename stem to numpy array."""
    if emb_dir is None:
        emb_dir = choose_embed_dir()
    if emb_dir is None:
        print('load_all_embeddings: no embedding directory found; returning empty dict')
        return {}
    files = sorted(list(Path(emb_dir).glob('*.npy')))
    if not files:
        print(f'No .npy files found in {emb_dir}')
        return {}
    emb = {}
    for p in files:
        try:
            arr = np.load(p)
        except Exception as exc:
            print(f'Could not load {p}: {exc}')
            continue
        if arr.ndim == 1:
            arr = arr.reshape(-1, 1)
        arr = arr.astype(np.float64)
        norms = np.linalg.norm(arr, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        arr = arr / norms
        emb[p.stem] = arr
    return emb

embeddings = load_all_embeddings()
print('Loaded embeddings:', list(embeddings.keys()))

def safe_silhouette_score(matrix, labels, sample_size=SILHOUETTE_SAMPLE, metric='cosine'):
    """Compute silhouette score robustly with optional sampling.

    Returns NaN for degenerate label sets (1 cluster or every point its own cluster).
    """
    unique = np.unique(labels)
    if len(unique) < 2 or len(unique) >= len(labels):
        return np.nan
    n = len(labels)
    try:
        if n > sample_size:
            rng = check_random_state(RANDOM_STATE)
            idx = rng.choice(n, sample_size, replace=False)
            return float(silhouette_score(matrix[idx], labels[idx], metric=metric))
        else:
            return float(silhouette_score(matrix, labels, metric=metric))
    except Exception as exc:
        print('Silhouette computation failed:', exc)
        return np.nan

def k_search_on_matrix(matrix, k_min=K_MIN, k_max=K_MAX, random_state=RANDOM_STATE):
    """Compute inertia (SSE) and silhouette for k in [k_min, k_max].

    Returns a dict with keys 'ks', 'sse' and 'silhouette'.
    """
    n_samples = matrix.shape[0]
    ks = list(range(k_min, min(k_max, n_samples - 1) + 1))
    sse = []
    sils = []
    for k in ks:
        try:
            km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
            labels = km.fit_predict(matrix)
            sse.append(float(km.inertia_))
            sils.append(safe_silhouette_score(matrix, labels))
        except Exception as exc:
            print(f'k={k} failed: {exc}')
            sse.append(np.nan)
            sils.append(np.nan)
    return {'ks': ks, 'sse': sse, 'silhouette': sils}

def plot_k_search(results, title, out_prefix):
    """Save elbow (inertia) and silhouette plots for a given k-search result."""
    ks = results['ks']
    sse = results['sse']
    sils = results['silhouette']

    plt.figure(figsize=(8, 3.5))
    plt.plot(ks, sse, marker='o')
    plt.xlabel('k')
    plt.ylabel('Inertia (SSE)')
    plt.title(f'{title} — Elbow')
    plt.grid(True)
    elbow_path = FIG_DIR / f'{out_prefix}_elbow.png'
    plt.savefig(elbow_path, dpi=150, bbox_inches='tight')
    plt.close()

    plt.figure(figsize=(8, 3.5))
    plt.plot(ks, sils, marker='o')
    plt.xlabel('k')
    plt.ylabel('Silhouette')
    plt.title(f'{title} — Silhouette')
    plt.grid(True)
    sil_path = FIG_DIR / f'{out_prefix}_silhouette.png'
    plt.savefig(sil_path, dpi=150, bbox_inches='tight')
    plt.close()

    return elbow_path, sil_path

def pick_k_by_silhouette(results):
    """Return k with maximum silhouette score; None if silhouettes are all NaN."""
    sils = np.array(results['silhouette'], dtype=np.float64)
    ks = np.array(results['ks'], dtype=int)
    if np.all(np.isnan(sils)):
        return None
    valid = ~np.isnan(sils)
    best_idx = np.argmax(sils[valid])
    return int(ks[valid][best_idx])


Loaded embeddings: ['baseline', 'd2v_line_d100_w5_dbow', 'd2v_line_d100_w5_dm', 'd2v_poem_d100_w5_dbow', 'd2v_poem_d100_w5_dm', 'w2v_d100_w5_cbow_avg', 'w2v_d100_w5_cbow_idf', 'w2v_d100_w5_cbow_sif', 'w2v_d100_w5_sg_avg', 'w2v_d100_w5_sg_idf', 'w2v_d100_w5_sg_sif']
